In [0]:
%sql
CREATE OR REPLACE TABLE dbw_maritime_2026.maritime_showcase.silver_ais_cleaned
AS
SELECT
  ingestion_time AS raw_ingestion_time,
  json_payload:MetaData.MMSI::string AS mmsi,
  json_payload:MetaData.ShipName::string AS ship_name,
  json_payload:MetaData.latitude::double AS latitude,
  json_payload:MetaData.longitude::double AS longitude,
  left(json_payload:MetaData.time_utc::string, 19)::timestamp AS event_time_utc
FROM dbw_maritime_2026.maritime_showcase.bronze_ais_raw
WHERE json_payload IS NOT NULL AND json_payload:MetaData.latitude IS NOT NULL AND json_payload:MetaData.longitude IS NOT NULL;

In [0]:
%sql
CREATE OR REPLACE TABLE dbw_maritime_2026.maritime_showcase.silver_ais_enriched AS
WITH ships_with_h3 AS (
  SELECT *, h3_longlatash3(longitude, latitude, 7) AS ship_h3_id_res7
  FROM dbw_maritime_2026.maritime_showcase.silver_ais_cleaned
)
SELECT s.*, p.port_name AS current_port_vicinity, p.country_code AS port_country
FROM ships_with_h3 s
LEFT JOIN dbw_maritime_2026.maritime_showcase.dim_world_ports p ON s.ship_h3_id_res7 = p.port_h3_id_res7;